# Scalable Topology Mining for Reward Spaces

This notebook implements the **Sheaf-Theoretic Reward Spaces** mining pipeline, adapted for Google Colab (T4/A100 GPU). 

It replaces dense matrix operations with **Sparse Spectral Methods** and **FAISS** for approximate nearest neighbors, allowing us to map the topology of the Anthropic HH-RLHF dataset (160k+ samples).

### Objectives
1. **Manifold Construction**: Build a semantic k-NN graph of prompts.
2. **Vector Field Embedding**: Map human preferences to high-dimensional reward vectors.
3. **Hodge Decomposition (Approximated)**: Identify regions of high "Harmonic Risk" (inconsistent preferences) by analyzing local vector field variance.
4. **Export**: Generate a `topology_metadata.parquet` map for downstream safety training.

In [ ]:
# @title 1. Setup & Dependencies
# Install fast approximate nearest neighbors (FAISS) and sparse solvers
!pip install -q sentence-transformers datasets faiss-gpu scipy networkx pyarrow

import torch
import numpy as np
import pandas as pd
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy import sparse
from scipy.sparse.linalg import lsqr
import networkx as nx
import gc
from tqdm.auto import tqdm

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Using a smaller, faster model for the mining phase to fit in Colab T4 memory
# Switch to 'gte-large' or 'all-mpnet-base-v2' if you have an A100
EMBEDDING_MODEL = "all-MiniLM-L6-v2" 
K_NEIGHBORS = 15  # Sparsity factor for the graph
BATCH_SIZE = 128
SAMPLE_LIMIT = 50000 # Subsample for demo speed/memory. Set to None for full run.

print(f"Running on {DEVICE}")

In [ ]:
# @title 2. Load & Encode Data (The Manifold Substrate)
# We load the Anthropic HH-RLHF dataset and extract the last turn
dataset = load_dataset("anthropic/hh-rlhf", split="train")

if SAMPLE_LIMIT:
    print(f"Subsampling to {SAMPLE_LIMIT} examples...")
    dataset = dataset.select(range(SAMPLE_LIMIT))

def extract_pairs(example):
    # Simple extraction of the final prompt and both responses
    # We assume the 'chosen' and 'rejected' share the same context/prompt
    # Split by the last "Assistant:" marker to separate Prompt from Response
    
    try:
        prompt = example["chosen"].rpartition("\n\nAssistant:")[0]
        chosen_response = example["chosen"].rpartition("\n\nAssistant:")[2]
        rejected_response = example["rejected"].rpartition("\n\nAssistant:")[2]
    except:
        # Fallback for parsing errors
        prompt = example["chosen"][:100]
        chosen_response = ""
        rejected_response = ""

    return {
        "prompt": prompt,
        "chosen_response": chosen_response,
        "rejected_response": rejected_response
    }

print("Preprocessing dataset...")
# Use num_proc=2 for Colab compatibility (limited multiprocessing)
processed_data = dataset.map(extract_pairs, num_proc=2)
prompts = processed_data["prompt"]

print("Loading Embedding Model...")
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

# 1. Encode Prompts (The Manifold Nodes)
print("Embedding prompts...")
prompt_embeddings = model.encode(prompts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(prompt_embeddings) # Cosine similarity = L2 on normalized vectors

print(f"Manifold Shape: {prompt_embeddings.shape}")

In [ ]:
# @title 3. Embed Preference Vectors (The Reward Field)
# To compute inconsistency, we need the "flow" or direction of preference.
# Vector = Embedding(Chosen) - Embedding(Rejected)

print("Embedding responses to compute preference vectors...")
# We encode chosen and rejected responses to get the preference direction vector
chosen_embs = model.encode(processed_data["chosen_response"], batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
rejected_embs = model.encode(processed_data["rejected_response"], batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)

# Calculate Preference Delta Vectors
preference_vectors = chosen_embs - rejected_embs

# Normalize to focus on direction, not magnitude (optional, but good for cosine analysis)
norms = np.linalg.norm(preference_vectors, axis=1, keepdims=True)
preference_directions = preference_vectors / (norms + 1e-8)

print("Preference field computed.")

# Clean up memory
del chosen_embs, rejected_embs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# @title 4. Construct the Preference Graph (Sparse)
# We build a k-NN graph based on semantic similarity of PROMPTS.
# Edges represent "semantically adjacent" situations.

print("Building k-NN graph with FAISS...")
# FAISS IndexFlatIP uses Inner Product (Cosine sim because normalized)
d = prompt_embeddings.shape[1]
index = faiss.IndexFlatIP(d)
if DEVICE == "cuda":
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)

index.add(prompt_embeddings)
D, I = index.search(prompt_embeddings, K_NEIGHBORS)

# I contains the indices of the neighbors for each node
# D contains the similarity scores
print(f"Graph constructed. Neighbors found for {len(I)} nodes.")

In [ ]:
# @title 5. Hodge Decomposition (Harmonic Risk Analysis)
# We calculate "Harmonic Risk" (Inconsistency) by analyzing the variance of 
# preference directions in the local neighborhood.

# Theoretical mapping:
# - In a consistent scalar reward field, Gradient(V) should explain preferences.
# - If similar states have divergent/contradictory preference vectors, 
#   Gradient(V) fails -> High Curl/Harmonic component.
# - We measure this as: 1 - mean_cosine_similarity(neighbors_vectors, mean_vector)

print("Calculating Harmonic Inconsistency (Hodge Residuals)...")

harmonic_risk_scores = np.zeros(len(prompt_embeddings))

# Vectorized computation using numpy indexing
# 1. Gather neighbor preference vectors
# Shape: (N_samples, K_neighbors, Embedding_dim)
neighbor_indices = I  # (N, K)
neighbor_vectors = preference_directions[neighbor_indices] 

# 2. Compute local mean vector (The "Smoothed Gradient")
local_mean_vectors = np.mean(neighbor_vectors, axis=1)
# Normalize mean vectors
mean_norms = np.linalg.norm(local_mean_vectors, axis=1, keepdims=True)
local_mean_dirs = local_mean_vectors / (mean_norms + 1e-8)

# 3. Compute consistency
# Dot product of each neighbor with the local mean
# We want to know: Do all neighbors agree with the consensus?
# Shape: (N, K)
consistencies = np.sum(neighbor_vectors * local_mean_dirs[:, np.newaxis, :], axis=2)

# Risk = 1 - Average Consistency
# If everyone aligns perfectly with mean -> Consistency 1.0 -> Risk 0.0
# If vectors are random/orthogonal -> Consistency ~0.0 -> Risk 1.0
# If vectors oppose each other -> Consistency < 0 -> Risk > 1.0
avg_consistency = np.mean(consistencies, axis=1)
harmonic_risk_scores = 1.0 - avg_consistency

# Normalize scores to 0-1 range for easier interpretation
min_risk = np.min(harmonic_risk_scores)
max_risk = np.max(harmonic_risk_scores)
harmonic_risk_scores = (harmonic_risk_scores - min_risk) / (max_risk - min_risk + 1e-8)

print(f"Analysis Complete.")
print(f"Mean Risk: {np.mean(harmonic_risk_scores):.4f}")
print(f"Max Risk: {np.max(harmonic_risk_scores):.4f}")

In [ ]:
# @title 6. Export Metadata
# We save the risk scores to be injected into the GeoDPO trainer.

import pyarrow as pa
import pyarrow.parquet as pq

# Create a DataFrame with the metadata
metadata_df = pd.DataFrame({
    "prompt": prompts,
    "harmonic_risk": harmonic_risk_scores, 
    "embedding_id": range(len(prompts)),
    # Optional: Save embeddings if needed downstream (careful with size)
    # "embedding": list(prompt_embeddings)
})

# Save to disk
metadata_df.to_parquet("topology_metadata.parquet")
print("Saved to topology_metadata.parquet")

# Visualize the top 5 "Black Holes" (High Risk Nodes)
print("\nTop 5 High-Risk (Inconsistent) Areas:")
top_risks = metadata_df.nlargest(5, "harmonic_risk")
for i, row in top_risks.iterrows():
    print(f"\n[Risk: {row['harmonic_risk']:.3f}] Prompt: {row['prompt'][:150]}...")